In [ ]:
#################################################################
##################### Valid Scenario Finder #####################
#################################################################

import sys, os
sys.path.insert(0, "nuplan-devkit")

from nuplan.planning.scenario_builder.nuplan_db.nuplan_scenario_builder import NuPlanScenarioBuilder
from nuplan.planning.scenario_builder.scenario_filter import ScenarioFilter
from nuplan.planning.utils.multithreading.worker_parallel import SingleMachineParallelExecutor

DATA_ROOT = "/home/user_218/shaditaymor/data/data1/nuplan-v1.1/splits/mini"
MAP_ROOT  = "/home/user_218/shaditaymor/data/data1/maps"

#####################################################
#####################################################
# PARAMS
TARGET_LOG = "2021.06.09.12.39.51_veh-26_05620_06003"  # change this to any log_name without .db
N = 1222333444555666  # how many runnable tokens to return
#####################################################
#####################################################

builder = NuPlanScenarioBuilder(
    data_root=DATA_ROOT,
    map_root=MAP_ROOT,
    sensor_root=None,
    db_files=None,
    map_version="nuplan-maps-v1.0",
    include_cameras=False,
)

sf = ScenarioFilter(
    scenario_types=None,
    scenario_tokens=None,
    log_names=[TARGET_LOG],
    map_names=None,
    num_scenarios_per_type=None,
    limit_total_scenarios=N,
    timestamp_threshold_s=None,
    ego_displacement_minimum_m=None,
    expand_scenarios=False,
    remove_invalid_goals=False,
    shuffle=False,
)

worker = SingleMachineParallelExecutor(use_process_pool=False, max_workers=4)
scenarios = builder.get_scenarios(sf, worker)

print("FOUND:", len(scenarios))
for s in scenarios:
    print(s.token, s.scenario_type, s.log_name)

In [ ]:
#################################################################
################### Scenario Validity Checker ###################
#################################################################

import sys
sys.path.insert(0, "nuplan-devkit")

from nuplan.planning.scenario_builder.nuplan_db.nuplan_scenario_builder import NuPlanScenarioBuilder
from nuplan.planning.scenario_builder.scenario_filter import ScenarioFilter
from nuplan.planning.utils.multithreading.worker_parallel import SingleMachineParallelExecutor

DATA_ROOT = "/home/user_218/shaditaymor/data/data1/nuplan-v1.1/splits/mini"
MAP_ROOT  = "/home/user_218/shaditaymor/data/data1/maps"

TOKENS = ["ffa83e1b737a5975", "ff1dd74e8d075989", "0020f438f6fb5860", "00435cc4f8d05d98"]

builder = NuPlanScenarioBuilder(
    data_root=DATA_ROOT,
    map_root=MAP_ROOT,
    sensor_root=None,
    db_files=None,
    map_version="nuplan-maps-v1.0",
    include_cameras=False,
)

sf = ScenarioFilter(
    scenario_types=None,
    scenario_tokens=TOKENS,
    log_names=None,
    map_names=None,
    num_scenarios_per_type=None,
    limit_total_scenarios=len(TOKENS),
    timestamp_threshold_s=None,
    ego_displacement_minimum_m=None,
    expand_scenarios=False,
    remove_invalid_goals=False,
    shuffle=False,
)

worker = SingleMachineParallelExecutor(use_process_pool=False, max_workers=4)
scenarios = builder.get_scenarios(sf, worker)

found = [s.token for s in scenarios]
missing = [t for t in TOKENS if t not in set(found)]

print("FOUND:", len(found), found)
print("MISSING:", len(missing), missing)
for s in scenarios:
    print("  token", s.token, "type", s.scenario_type, "log", s.log_name)

In [ ]:
#################################################################
####################### Simulation Runner #######################
#################################################################


import sys
import os
from datetime import datetime
import nest_asyncio
nest_asyncio.apply()

os.environ["HYDRA_FULL_ERROR"] = "1"

now = datetime.now()
# The format string uses codes for Year, Month, Day, Hour (24h), Minute, and Second
formatted_datetime = now.strftime("%Y%m%d%H%M%S")

print(formatted_datetime)

devkit_path = "nuplan-devkit"
planner_path = "Diffusion-Planner"

if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)
if planner_path not in sys.path:
    sys.path.insert(0, planner_path)

print(f"Updated sys.path:\n{sys.path}\n")
os.environ["NUPLAN_EXP_ROOT"] = "/home/user_218/shaditaymor/nuplan_results"  # pick your path
os.makedirs(os.environ["NUPLAN_EXP_ROOT"], exist_ok=True)


TOKENS = [
    "ffa83e1b737a5975", "ff1dd74e8d075989",
    "0020f438f6fb5860", "00435cc4f8d05d98"
]
scenario_tokens_override = (
    "scenario_filter.scenario_tokens=["
    + ", ".join(f"\"{t}\"" for t in TOKENS)
    + "]"
)

try:
    # Import the main function
    from nuplan.planning.script.run_simulation import main as run_simulation_main

    # Define the arguments 
    script_path = 'nuplan-devkit/nuplan/planning/script/run_simulation.py'
    args = [
        script_path, 
        f"experiment_name=testing_{formatted_datetime}",
        "scenario_builder=nuplan_mini",
        "scenario_builder.map_root=/home/user_218/shaditaymor/data/data1/maps",
        "scenario_builder.data_root=/home/user_218/shaditaymor/data/data1/nuplan-v1.1/splits/mini", # Kept as absolute path
        "+simulation=closed_loop_nonreactive_agents",
        "planner=diffusion_planner",
        "planner.diffusion_planner.config.args_file=/home/user_218/shaditaymor/Project-diffusion/Diffusion-Planner/checkpoints/args.json",
        "planner.diffusion_planner.ckpt_path=/home/user_218/shaditaymor/Project-diffusion/Diffusion-Planner/checkpoints/model.pth",

        "scenario_filter.shuffle=false",
        "scenario_filter.scenario_types=null",
        "scenario_filter.remove_invalid_goals=false",
        "scenario_filter.log_names=null",
        "scenario_filter.map_names=null",
        "scenario_filter.num_scenarios_per_type=null",
        "scenario_filter.timestamp_threshold_s=null",
        "scenario_filter.ego_displacement_minimum_m=null",
        "scenario_filter.expand_scenarios=false",
        "scenario_filter.limit_total_scenarios=4",

        "worker=sequential",
        "verbose=true",
        "hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter,pkg://diffusion_planner.config,pkg://nuplan.planning.script.config.common,pkg://nuplan.planning.script.experiments]"
    ]
    args.append(scenario_tokens_override)

    # Run the simulation
    print("Backing up original sys.argv...")
    original_argv = list(sys.argv)
    
    try:
        print("Setting new sys.argv for hydra...")
        sys.argv = args
        print(f"Running simulation with args: {sys.argv}")
        
        # Call the hydra-decorated main function
        run_simulation_main()
        
        print("\nSimulation finished.")
        
    except Exception as e:
        print(f"\nAn error occurred during simulation: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        # Restore the original sys.argv so it doesn't affect other notebook cells
        print("Restoring original sys.argv...")
        sys.argv = original_argv

except ImportError as e:
    print(f"Error: Failed to import modules: {e}")
    print("Please double-check the paths set in this cell:")
    print(f"Devkit path: {devkit_path}")
    print(f"Planner path: {planner_path}")
    print("Ensure these directories exist and contain the correct packages.")




In [ ]:
#################################################################
####################### NuBoard Visualize #######################
#################################################################

import sys
import os
import glob
import nest_asyncio
import socket
from bokeh.io import curdoc



def _pick_free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port

nest_asyncio.apply()

# Make sure your python can import nuplan-devkit
devkit_path = 'nuplan-devkit'
if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)

from nuplan.planning.script.run_nuboard import main as run_nuboard_main

# ----------------------------
# Settings
# ----------------------------
PORT = _pick_free_port()
MAX_RUNS = 10  # load last N experiments
NUBOARD_GLOB = "/home/user_218/shaditaymor/nuplan_results/exp/*/*/*/nuboard_*.nuboard"

# Find newest .nuboard files
nuboard_files = sorted(glob.glob(NUBOARD_GLOB), key=os.path.getmtime, reverse=True)[:MAX_RUNS]
if not nuboard_files:
    raise RuntimeError(f"No .nuboard files found with pattern: {NUBOARD_GLOB}")

# Hydra list formatting: simulation_path=[file1,file2,...]
simulation_path_override = (
    "simulation_path=["
    + ", ".join(f"\"{p}\"" for p in nuboard_files)
    + "]"
)

print("Loading nuBoard files:")
for f in nuboard_files:
    print("  -", f)

# Build argv for hydra
script_path = "/home/user_218/shaditaymor/Project-diffusion/nuplan-devkit/nuplan/planning/script/run_nuboard.py"
args = [
    script_path,
    f"port_number={PORT}",
    "worker=sequential",
    simulation_path_override,
    "scenario_builder.data_root=/home/user_218/shaditaymor/data/data1/nuplan-v1.1/splits/mini",
    "scenario_builder.map_root=/home/user_218/shaditaymor/data/data1/maps",
]

# Run
curdoc().clear()
original_argv = list(sys.argv)
try:
    sys.argv = args
    print("\nStarting nuBoard with args:\n", "\n".join(sys.argv))
    print(f"nuBoard URL: http://localhost:{PORT}/")
    run_nuboard_main()
finally:
    sys.argv = original_argv
